# 01 — Análise Exploratória do Dataset (EuroSAT RGB)

**Global Solution — Visão Computacional aplicada à Indústria Espacial**

Classificação de uso e cobertura do solo (LULC) a partir de imagens do satélite **Sentinel-2** (dataset EuroSAT RGB). O resultado da CNN é o componente de visão computacional de uma plataforma que agrega dados meteorológicos de sensoriamento remoto.

Este notebook: carrega o dataset, inspeciona as classes, visualiza amostras e define a divisão estratificada treino/validação/teste.

In [ ]:
# >>> NO GOOGLE COLAB: rode ESTA célula primeiro (em ambiente local, pule) <<<
# Clona o repositório. NÃO instalamos requirements.txt no Colab: ele já traz
# TensorFlow, Keras, numpy, pandas, scikit-learn, matplotlib, seaborn e
# tensorflow-datasets em versões compatíveis entre si. Reinstalar versões
# fixas rebaixa o ml_dtypes/numpy e quebra o JAX do Colab.
# !git clone https://github.com/luanmacea/gs-cnn-eurosat.git
# %cd gs-cnn-eurosat

In [ ]:
import sys, os
# Resolve a raiz do projeto de forma robusta (local OU Colab, com ou sem
# restart de runtime). Procura uma pasta que contenha 'src/'.
candidatos = [os.getcwd(), os.path.dirname(os.getcwd()), '/content/gs-cnn-eurosat']
ROOT = next((p for p in candidatos if os.path.isdir(os.path.join(p, 'src'))), os.getcwd())
os.chdir(ROOT)  # paths relativos (models/, reports/) consistentes
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print('Project root:', ROOT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from src.data_loader import load_raw_data, make_splits, CLASS_NAMES, IMG_SIZE, NUM_CLASSES
sns.set_theme(style='whitegrid')

## Carregamento
Na primeira execução o TensorFlow Datasets baixa o EuroSAT (~90 MB) e o mantém em cache. As imagens são carregadas como `uint8 [0,255]` — a normalização acontece dentro do modelo (camada `Rescaling`).

In [ ]:
images, labels = load_raw_data()
print('Imagens:', images.shape, images.dtype)
print('Labels :', labels.shape, '| nº de classes:', NUM_CLASSES)
print('Classes:', CLASS_NAMES)

## Distribuição de imagens por classe

In [ ]:
counts = Counter(labels.tolist())
plt.figure(figsize=(11, 4))
plt.bar([CLASS_NAMES[i] for i in range(NUM_CLASSES)],
        [counts[i] for i in range(NUM_CLASSES)], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.ylabel('nº de imagens'); plt.title('Distribuição de imagens por classe')
plt.tight_layout(); plt.show()
for i in range(NUM_CLASSES):
    print(f'{CLASS_NAMES[i]:24s}: {counts[i]}')

O EuroSAT é levemente desbalanceado (entre 2.000 e 3.000 imagens por classe). A divisão estratificada preserva essa proporção em treino, validação e teste.

## Amostras de cada classe

In [ ]:
n_cols = 5
fig, axes = plt.subplots(NUM_CLASSES, n_cols, figsize=(10, 2 * NUM_CLASSES))
for c in range(NUM_CLASSES):
    idx = np.where(labels == c)[0][:n_cols]
    for j, k in enumerate(idx):
        axes[c, j].imshow(images[k])
        axes[c, j].axis('off')
    axes[c, 0].set_title(CLASS_NAMES[c], loc='left', fontsize=10)
plt.tight_layout(); plt.show()

## Estatísticas por canal (R, G, B)

In [ ]:
flat = images.reshape(-1, 3).astype('float32')
for ci, cn in enumerate(['R', 'G', 'B']):
    print(f'Canal {cn}: média={flat[:, ci].mean():6.2f} | desvio={flat[:, ci].std():6.2f}')
print('\nValores em [0,255] -> por isso o modelo aplica Rescaling(1/255) na 1ª camada.')

## Divisão treino / validação / teste (estratificada — 70/15/15)
`seed=42` fixa a divisão para reprodutibilidade.

In [ ]:
splits = make_splits(images, labels, seed=42)
for name, (x, y) in splits.items():
    print(f'{name:6s}: {len(x):6d} imagens')

print('\nProporção de classes por split (deve ser ~igual):')
for name, (x, y) in splits.items():
    dist = np.bincount(y, minlength=NUM_CLASSES) / len(y)
    print(f'{name:6s}:', np.round(dist, 3))

**Conclusão da EDA.** Dataset com 10 classes de uso do solo, levemente desbalanceado, imagens 64×64 RGB. A divisão estratificada mantém o balanceamento. Próximo passo: treinar a CNN-A (baseline) no notebook 02.